# 🇯🇵 Nemesis — Japan Daily Stock Recommendation

**Data Sources:**
- 📊 **J-Quants Pro** — OHLCV, 信用倍率, 空売り残高, 投資主体別売買動向
- 📋 **EDINET** — 大量保有報告書 (≥5%), 臨時報告書
- 📣 **TDnet (Yanoshin)** — 適時開示 (earnings revisions, M&A, dividends)
- 🌐 **US Overnight (yfinance)** — S&P500/Nasdaq futures, VIX, USD/JPY, sector ETFs
- 🏦 **Bank of Japan API** — policy rate, JGB yields, M2
- 📈 **e-Stats** — CPI, unemployment, retail sales
- 🏭 **METI IIP** — industrial production by sector

**Strategies:**
- `multi_factor`: Normal days — weighted combination of all signals
- `shock_recovery`: After US drops >1% — buy oversold JP stocks with no bad news

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) != 'Nemesis' else os.getcwd())

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')

from datetime import date, datetime
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

print('✅ Setup complete')

In [ ]:
# ─── Configure target date ───────────────────────────────────────────
# Set to 'today' for live run, or a specific date for historical analysis
TARGET_DATE = date.today()          # Live
# TARGET_DATE = date(2025, 3, 18)  # Historical

TOP_N = 30  # Number of top recommendations to display

print(f'Target date: {TARGET_DATE} ({TARGET_DATE.strftime("%A")})')

## Step 1: US Overnight Market Check
*(Run this first — determines which strategy to use)*

In [ ]:
from japan_stock_daily.collectors.us_overnight_collector import USOverNightCollector

us_collector = USOverNightCollector()
us_data = us_collector.get_us_overnight_performance()

print(f"Market Bias: {us_data.get('market_bias', 'unknown').upper()}")
print(f"S&P500 Futures: {us_data.get('sp500_futures_chg', 'N/A'):+.2f}%"  
      if us_data.get('sp500_futures_chg') else "S&P500 Futures: N/A")
print(f"Nasdaq Futures: {us_data.get('nasdaq_futures_chg', 'N/A'):+.2f}%"
      if us_data.get('nasdaq_futures_chg') else "Nasdaq Futures: N/A")
print(f"VIX: {us_data.get('vix', 'N/A')}")
print(f"USD/JPY: {us_data.get('usd_jpy', 'N/A')} ({us_data.get('yen_direction', '')})")
print()
print('Sector ETF Overnight Performance:')
for etf, chg in (us_data.get('sector_signals') or {}).items():
    if chg is not None:
        arrow = '▲' if chg > 0 else '▼'
        print(f"  {etf}: {arrow} {chg:+.2f}%")

## Step 2: Full Daily Scoring Pipeline

In [ ]:
from japan_stock_daily.recommender.scorer import DailyScorer

scorer = DailyScorer()
recommendations = scorer.score_all(TARGET_DATE)

print(f'\nTotal stocks scored: {len(recommendations)}')
print(f'Strategy used: {recommendations["strategy"].iloc[0] if not recommendations.empty else "N/A"}')

In [ ]:
# Display top recommendations
top_recs = scorer.get_top_recommendations(recommendations, n=TOP_N)

display_cols = [
    'rank', 'code', 'name', 'sector33',
    'composite_score', 'disc_score', 'sd_score', 'us_score', 'momentum_score', 'macro_score',
    'disc_category', 'strategy'
]
print(f'\nTop {TOP_N} Recommendations for {TARGET_DATE}:')
top_recs[[c for c in display_cols if c in top_recs.columns]]

In [ ]:
# Show signal details for top 10
print('\nTop 10 Signal Details:')
for _, row in top_recs.head(10).iterrows():
    print(f"\n#{int(row['rank'])} {row['code']} {row.get('name','')[:20]} [{row['sector33']}]")
    print(f"  Score: {row['composite_score']:.1f} | Disc:{row['disc_score']:.0f} SD:{row['sd_score']:.0f} US:{row['us_score']:.0f}")
    if row.get('disc_signals'):
        print(f"  {row['disc_signals'][:150]}")
    if row.get('sd_signals'):
        print(f"  {row['sd_signals'][:100]}")
    if row.get('us_signals'):
        print(f"  {row['us_signals'][:100]}")

## Step 3: TDnet Disclosure Highlights

In [ ]:
from japan_stock_daily.collectors.tdnet_collector import TDnetCollector

tdnet = TDnetCollector()
tdnet_df = tdnet.get_daily_disclosures(TARGET_DATE)

if not tdnet_df.empty:
    print(f'TDnet disclosures today: {len(tdnet_df)}')
    print('\nHigh-impact disclosures (score >= 60):')
    high_impact = tdnet_df[tdnet_df['category_score'] >= 60]
    for _, row in high_impact.iterrows():
        print(f"  [{row['category_score']:.0f}] {row['code']} {row['company']}: {row['title'][:80]}")
else:
    print('No TDnet data available')

## Step 4: EDINET Disclosure Signals

In [ ]:
from japan_stock_daily.collectors.edinet_collector import EdinetCollector

edinet = EdinetCollector()
edinet_df = edinet.get_daily_signals(TARGET_DATE)

if not edinet_df.empty:
    print(f'EDINET signals today: {len(edinet_df)}')
    print()
    # Large shareholder reports
    large_holders = edinet_df[edinet_df['doc_type_code'].isin(['160', '161'])]
    if not large_holders.empty:
        print('大量保有報告書 (Large Shareholder ≥5%):')
        for _, row in large_holders.iterrows():
            print(f"  {row.get('code','')} {row.get('filer_name','')[:30]} → {row.get('direction','')} "
                  f"({row.get('ownership_pct','')}%)")
    # Extraordinary reports
    rinji = edinet_df[edinet_df['doc_type_code'] == '140']
    if not rinji.empty:
        print('\n臨時報告書 (Extraordinary Reports):')
        for _, row in rinji.iterrows():
            print(f"  {row.get('code','')} {row.get('filer_name','')[:30]} [{row.get('event_category','')}]")
else:
    print('No EDINET signals today')

## Step 5: Macroeconomic Context

In [ ]:
from japan_stock_daily.collectors.boj_collector import BOJCollector
from japan_stock_daily.collectors.estats_collector import EStatsCollector

boj = BOJCollector()
estats = EStatsCollector()

boj_data = boj.get_all_indicators()
estats_data = estats.get_all_indicators()

print('=== Bank of Japan ===')
for k, v in boj_data.items():
    print(f'  {k}: {v}')
print(f'  Rate environment: {boj.get_rate_environment()}')

print('\n=== e-Stats Government Statistics ===')
for k, v in estats_data.items():
    print(f'  {k}: {v}')

## Step 6: Generate HTML Report

In [ ]:
from japan_stock_daily.reports.report_generator import ReportGenerator

report_gen = ReportGenerator()
html_path = report_gen.generate(
    target_date=TARGET_DATE,
    recommendations=recommendations,
    us_data=us_data,
    boj_data=boj_data,
    estats_data=estats_data,
    tdnet_df=tdnet_df if 'tdnet_df' in dir() else None,
    edinet_df=edinet_df if 'edinet_df' in dir() else None,
    top_n=TOP_N,
)

print(f'\nReport generated: {html_path}')

# Display link
from IPython.display import HTML
HTML(f'<a href="{html_path}" target="_blank">📊 Open Report in Browser</a>')

## Deep Dive: Single Stock Analysis

In [ ]:
# ─── Enter stock code to deep-dive ───────────────────────────────────
DEEP_DIVE_CODE = recommendations['code'].iloc[0] if not recommendations.empty else '7203'

row = recommendations[recommendations['code'] == DEEP_DIVE_CODE]
if not row.empty:
    r = row.iloc[0]
    print(f'=== Deep Dive: {DEEP_DIVE_CODE} {r.get("name", "")} ===')
    print(f'Sector: {r.get("sector33", "")} | Market: {r.get("market", "")}')
    print(f'Composite Score: {r["composite_score"]:.1f}')
    print(f'Strategy: {r.get("strategy", "")}')
    print()
    print(f'Disclosure Score: {r["disc_score"]:.0f}')
    if r.get('disc_signals'):
        for sig in str(r['disc_signals']).split(' | '):
            if sig: print(f'  {sig}')
    print(f'\nSupply/Demand Score: {r["sd_score"]:.0f}')
    if r.get('sd_signals'):
        for sig in str(r['sd_signals']).split('; '):
            if sig: print(f'  {sig}')
    print(f'\nUS Overnight Score: {r["us_score"]:.0f} (Direction: {r.get("us_expected_dir", "")})')
    if r.get('us_signals'):
        for sig in str(r['us_signals']).split(' | '):
            if sig: print(f'  {sig}')
    print(f'\nMomentum Score: {r["momentum_score"]:.0f}')
    print(f'Macro Score: {r["macro_score"]:.0f}')
    if r.get('macro_signals'):
        for sig in str(r['macro_signals']).split(' | '):
            if sig: print(f'  {sig}')
else:
    print(f'Code {DEEP_DIVE_CODE} not found in recommendations')